In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

spark = (
    SparkSession.builder
    .appName("SparkSQL_Class")
    .config("spark.sql.pushdown.enabled", "true")
    .getOrCreate()
)

In [0]:
churn_cleaned = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true") 
    .csv("/Workspace/Users/ana.marta.dias.bs@gmail.com/big_data_analysis_project/churn_cleaned.csv")
)

In [0]:
churn_cleaned.show()

Overall churn rate

In [0]:
churn_rate = churn_cleaned.agg(
    F.count("*").alias("total"),
    F.sum("Exited").alias("churned"),
    F.round(F.avg("Exited") * 100, 2).alias("churn_pct")
)
churn_rate.show()

There is a class imbalance (only around 20% of the customers churned), that should be taken into account for our model

Churn rate by category

In [0]:
for col in ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember"]:
    churn_cleaned.groupBy(col).agg(
        F.count("*").alias("n"),
        F.round(F.avg("Exited") * 100, 2).alias("churn_pct")
    ).orderBy(col).show()

- German customers are the most likely to churn
- Female customers have a mildly higher churn-rate
- NumOfProducts is a massive predictor, customers with 2 products are very likely to stay, the ones with 3 are very likely to churn, and everyone with 4 products churns
- HasCrCard has practically no effect on the target
- Active members are roughly half as likely to churn as inactive ones

Visualize distributions

In [0]:
sample_pd = churn_cleaned.select("CreditScore","Age","Balance","Tenure","EstimatedSalary","Exited").sample(0.3).toPandas()

fig, axes = plt.subplots(2, 3, figsize=(16, 13))
for ax, c in zip(axes.ravel(), ["CreditScore","Age","Balance","Tenure","EstimatedSalary"]):
    sns.histplot(data=sample_pd, x=c, hue="Exited", kde=True, ax=ax, stat="density", common_norm=False)
    ax.set_title(f"{c} distribution by churn")
axes.ravel()[-1].axis("off")
plt.tight_layout
plt.show()

- Age is right-skewed, churners peak at 40-50 vs a young retained base
- Balance has a large spike at 0 (non-churners), while churners concentrate around 100k-150k
- CreditScore follows a typical normal distribution
- Tenure and EstimatedSalary are mostly uniform and have weak standalone signals (neither of these features is a strong churn predictor)

Correlation heatmap of the numeric variable

In [0]:
corr_pd = churn_cleaned.select(numeric_cols).toPandas().corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_pd, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap (numeric variables incl. target)")
plt.tight_layout(); plt.show()

The strongest correlations are 
- Age + Exited (0.29): As age increses, the probability of churning increases
- Balance + NumOfProducts (-0.30): As NumOfProducts increases, Balance tends to decrease

Compare statistics of Churners vs Stayers 

In [0]:
agg_df = churn_cleaned.groupBy("Exited").agg(
    F.round(F.avg("Age"), 1).alias("avg_age"),
    F.round(F.avg("Balance"), 0).alias("avg_balance"),
    F.round(F.avg("CreditScore"), 0).alias("avg_credit"),
    F.round(F.avg("Tenure"), 1).alias("avg_tenure"),
    F.round(F.avg("NumOfProducts"), 2).alias("avg_products"),
    F.round(F.avg("EstimatedSalary"), 0).alias("avg_salary"),
    F.round(F.avg("IsActiveMember") * 100, 1).alias("pct_active"),
    F.round(F.avg("HasCrCard") * 100, 1).alias("pct_has_card")
)
final_df = agg_df.orderBy("Exited")
final_df.show()

In [0]:
# mean comparison plot
# CreditScore is scaled x100 and Age is scaled /10 for visualization purposes
features = ["EstimatedSalary", "Balance", "CreditScore", "NumOfProducts", "Age", "Tenure", "HasCrCard", "IsActiveMember", "Exited"]
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
means1 = churn_cleaned.groupBy("Exited").avg(*features[:3]).toPandas().set_index("Exited")
means1["avg(CreditScore)"] = means1["avg(CreditScore)"] * 100

means2 = churn_cleaned.groupBy("Exited").avg(*features[3:-1]).toPandas().set_index("Exited")
means2["avg(Age)"] = means2["avg(Age)"] / 10
means1.T.plot(kind="barh", ax=axes[0])    
means2.T.plot(kind="barh", ax=axes[1])

fig.suptitle("Mean feature values: stayed (0) vs churned (1)")
axes[0].legend(title="Exited", loc="lower right")
axes[1].legend(title="Exited", loc="lower right")
plt.tight_layout()
plt.show()

# boxplots
# Box plots aren't informative for binary variables, so IsActiveMember and HasCrCard don't have to be included in this set of graphs
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, c in zip(axes.ravel(), features[:-3]):
    sns.boxplot(data=sample_pd, x="Exited", y=c, ax=ax)
    ax.set_title(c)

for ax in axes.ravel()[6:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

Strong signals:
- Age: Churned customers are significantly older on average than those who stayed.

- IsActiveMember: Active members are more likely to stay, and inactive members are more likely to churn.

- Balance: We can see in the bar plot that customers who churn have a higher average balance. However, the box plot shows that the 25th percentile is exactly 0 for both groups, which reveals again that this feature is highly skewed. Among those with non-zero balances, the churned group still has a higher median balance than the stayed group, but a new boolean feature like Has_Balance might be needed to help our model.

Weak signals:
- EstimatedSalary: Salary has very little impact on whether someone churns.

- CreditScore: Credit score has very little impact on churn.

- Tenure: How long someone has been a customer doesn't strongly predict if they will leave.

- HasCrCard: Having a credit card doesn't seem to matter.

- NumOfProducts: While the orange bar (stayed) is barely longer than the blue one (churned), NumOfProducts is usually a non-linear signal: people with 1 product churn differently than people with 2, 3 or 4, but taking the average hides this relationship.

Multi-dimensional view of customers most likely to churn

In [0]:
churn_cleaned.withColumn(
    "age_band",
    F.when(F.col("Age") < 30, "<30")
     .when(F.col("Age") < 40, "30-39")
     .when(F.col("Age") < 50, "40-49")
     .otherwise("50+")
) \
.groupBy("Geography", "IsActiveMember", "age_band") \
.agg(
    F.count("*").alias("n"),
    F.round(F.avg("Exited") * 100, 2).alias("churn_pct")
) \
.orderBy(F.col("churn_pct").desc()) \
.show(15)

Inactive, 50+-year-olds show the highest churn-rates, especially from Germany.

In [0]:
data = [
    {"Indicator": "NumOfProducts", "Direction": "1 or 3–4 = high churn", "Business meaning": "Product-fit / servicing issue"},
    {"Indicator": "IsActiveMember = 0", "Direction": "higher churn", "Business meaning": "Disengaged customers leave"},
    {"Indicator": "Age (50+)", "Direction": "higher churn", "Business meaning": "Elder customers churn more"},
    {"Indicator": "Geography = Germany", "Direction": "higher churn", "Business meaning": "Market-specific dissatisfaction"},
    {"Indicator": "Balance > 0 (esp. high)", "Direction": "higher churn", "Business meaning": "High-value customers at risk"},
    {"Indicator": "Gender = Female", "Direction": "slightly higher churn", "Business meaning": "Secondary/weak"},
    {"Indicator": "Tenure, CreditScore, HasCrCard, EstimatedSalary", "Direction": "≈ flat", "Business meaning": "Not useful alone"}
]

df_indicators = pd.DataFrame(data)

# Display with styling
display(
    df_indicators.style
    .hide(axis="index")
    .set_properties(**{'text-align': 'left'})
    .set_properties(subset=['Direction'], **{'text-align': 'center'})
    # Add padding to the left and right of all cells
    .set_table_styles([
        {'selector': 'th, td', 'props': [('padding-left', '20px'), ('padding-right', '20px')]}
    ])
)